# DriveSense-VLM — 01: SFT Training

Fine-tunes Qwen3-VL-2B-Instruct with LoRA (rank 32, alpha 64) on the
DriveSense hazard detection dataset.

**GPU required**: A100 80GB  
**Estimated time**: ~6 hours  
**Estimated cost**: ~72 CU  

**Prerequisites**: Run `00_data_pipeline.ipynb` first to generate
`outputs/data/sft_ready/sft_train.jsonl`.

## ⚠️ Before Running

1. Click **Runtime → Change runtime type**
2. Set **Hardware accelerator** → **A100 GPU**
3. Click **Save**

Without an A100 (80 GB VRAM), training will OOM on the 2B parameter model
with LoRA + gradient checkpointing.

In [ ]:
# Cell 3: Mount Drive + clone repo + symlinks
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/DriveSense-VLM'
os.makedirs(PROJECT_ROOT, exist_ok=True)

!git clone https://github.com/jayanth922/DriveSense-VLM.git /content/DriveSense-VLM 2>/dev/null || (cd /content/DriveSense-VLM && git pull --quiet)
%cd /content/DriveSense-VLM

!mkdir -p {PROJECT_ROOT}/data {PROJECT_ROOT}/outputs
!ln -sf {PROJECT_ROOT}/data /content/DriveSense-VLM/data 2>/dev/null || true
!ln -sf {PROJECT_ROOT}/outputs /content/DriveSense-VLM/outputs 2>/dev/null || true

print('Setup complete.')
print(f'Project root (Drive): {PROJECT_ROOT}')

In [ ]:
# Cell 4: Install training dependencies
# flash-attn build takes ~5 min but dramatically reduces VRAM usage
!pip install -e '.[training]' -q
!pip install flash-attn --no-build-isolation -q
print('Training dependencies installed.')

In [ ]:
# Cell 5: Verify GPU
import torch

assert torch.cuda.is_available(), (
    'No GPU detected!\n'
    'Fix: Runtime → Change runtime type → A100 GPU'
)

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f'GPU  : {gpu_name}')
print(f'VRAM : {vram_gb:.1f} GB')

if 'A100' not in gpu_name:
    print(f'WARNING: Expected A100 but got {gpu_name}.')
    print('Training may fail with OOM on GPUs < 40 GB.')
else:
    print('A100 confirmed ✓')

In [ ]:
# Cell 6: Check for existing checkpoints (resume support)
import glob
from pathlib import Path

training_dir = str(Path(PROJECT_ROOT) / 'outputs' / 'training')
checkpoints = sorted(glob.glob(f'{training_dir}/checkpoint-*'))

if checkpoints:
    print(f'Found {len(checkpoints)} checkpoint(s):')
    for ckpt in checkpoints:
        print(f'  {ckpt}')
    print(f'\nLatest: {checkpoints[-1]}')
    print('Training will RESUME from this checkpoint automatically.')
else:
    print('No checkpoints found — starting fresh training.')

# Verify SFT data is present
sft_train = Path(PROJECT_ROOT) / 'outputs' / 'data' / 'sft_ready' / 'sft_train.jsonl'
if sft_train.exists():
    with open(sft_train) as f:
        n = sum(1 for _ in f)
    print(f'\nTraining data: {sft_train} ({n} examples) ✓')
else:
    raise FileNotFoundError(
        f'Training data not found: {sft_train}\n'
        'Run 00_data_pipeline.ipynb first.'
    )

In [ ]:
# Cell 7: Configure Weights & Biases
import wandb
wandb.login()  # Prompts for API key — get it at https://wandb.ai/authorize
print('W&B configured. Training metrics will be logged to project: drivesense-vlm')

## Start Training

Training saves checkpoints to Google Drive every 250 steps.
If the session disconnects, rerun Cells 3–4 (setup), then rerun
the cell below — it will automatically resume from the latest checkpoint.

In [ ]:
# Cell 9: Run SFT training
# RECOVERY: If Colab disconnects, just rerun this cell — auto-resumes from Drive checkpoint

# Copy SFT data to fast ephemeral storage for faster DataLoader I/O
!mkdir -p /content/sft_data
!cp {PROJECT_ROOT}/outputs/data/sft_ready/sft_train.jsonl /content/sft_data/
!cp {PROJECT_ROOT}/outputs/data/sft_ready/sft_val.jsonl /content/sft_data/
print('Training data copied to ephemeral /content/sft_data/ for fast I/O.')

!python scripts/run_training.py \
    --config configs/training.yaml \
    --override training.output_dir={PROJECT_ROOT}/outputs/training \
    --override training.save_steps=250 \
    --override training.save_total_limit=3 \
    --resume

## Training Results

In [ ]:
# Cell 11: Display training metrics
import json
import os
from pathlib import Path

metrics_path = Path(PROJECT_ROOT) / 'outputs' / 'training' / 'training_metrics.json'
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    print('Training Metrics:')
    print('-' * 40)
    for k, v in metrics.items():
        print(f'  {k:<30}: {v}')
else:
    print(f'Metrics file not found at {metrics_path}')
    print('Training may still be running or did not complete.')

In [ ]:
# Cell 12: Verify adapter saved to Drive
from pathlib import Path

training_dir = Path(PROJECT_ROOT) / 'outputs' / 'training'

# Check for best checkpoint
adapter_path = training_dir / 'checkpoint-best'
if not adapter_path.exists():
    # Fall back to latest numbered checkpoint
    import glob
    ckpts = sorted(glob.glob(str(training_dir / 'checkpoint-*')))
    if ckpts:
        adapter_path = Path(ckpts[-1])

assert adapter_path.exists(), (
    f'No checkpoint found in {training_dir}. Training may have failed.'
)

print(f'LoRA adapter saved at: {adapter_path}')
!ls -la {adapter_path}
print('\nProceed to 02_optimization.ipynb')